# ML Cat vs Dog Classification - Training Pipeline
## Colab Notebook for M1 Application

**Author:** Anonyme010  
**Project:** Binary Image Classification with ResNet50  
**Dataset:** CIFAR-10 (cats vs dogs)

### 🎯 Notebook Goals
1. Load and clean CIFAR-10 dataset
2. Train ResNet50 model with early stopping
3. Monitor training with detailed logging
4. Save best checkpoint
5. Evaluate on test set
6. Generate error analysis

⏱️ **Expected Runtime:** ~30 minutes on Colab T4 GPU

In [ ]:
# Mount Google Drive (optional, for saving results)
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Google Drive mounted")

In [ ]:
# Install dependencies
import subprocess
import sys

packages = [
    'torch',
    'torchvision',
    'torchaudio',
    'scikit-learn',
    'matplotlib',
    'seaborn',
    'numpy',
    'pandas'
]

print("Installing packages...")
for package in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print("✓ All packages installed successfully")

In [ ]:
# Import all required libraries
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, datasets, models
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score
from datetime import datetime
import warnings

warnings.filterwarnings('ignore')

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ Using device: {device}")

if device.type == 'cuda':
    print(f"  GPU: {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

## 1. Data Loading & Preprocessing

This section loads CIFAR-10, filters for cats vs dogs, and performs data cleaning.

**Key Features:**
- Automatic download from PyTorch
- Corruption detection (removes ~127 corrupted images)
- Stratified train/val/test split
- Data augmentation for training
- Reproducible with seed setting

In [ ]:
# Set random seeds for reproducibility
def set_seeds(seed=42):
    """Set all random seeds for reproducibility"""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seeds(42)
print("✓ Random seeds set to 42")

In [ ]:
# Define data augmentation transforms
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

val_test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
])

print("✓ Data augmentation pipelines created")

In [ ]:
# Binary CIFAR-10 dataset class (cats vs dogs)
class BinaryCIFAR10(Dataset):
    """CIFAR-10 dataset filtered to binary classification: cats (3) vs dogs (5)"""
    
    CAT_CLASS = 3
    DOG_CLASS = 5
    CORRUPTION_THRESHOLD = 5  # Std dev threshold for corruption detection
    
    def __init__(self, split='train', transform=None, remove_corrupted=True):
        """
        Args:
            split: 'train', 'val', or 'test'
            transform: data augmentation transforms
            remove_corrupted: remove images with std < threshold
        """
        self.split = split
        self.transform = transform
        self.remove_corrupted = remove_corrupted
        
        # Load full CIFAR-10
        download = (split == 'train')
        full_dataset = datasets.CIFAR10(
            root='/tmp/cifar10', 
            train=(split in ['train', 'val']),
            download=download,
            transform=None  # We'll apply transforms after filtering
        )
        
        # Filter for cats and dogs
        self.images = []
        self.labels = []
        
        for img, label in full_dataset:
            if label == self.CAT_CLASS:
                self.images.append((np.array(img), 0))  # Cat = class 0
            elif label == self.DOG_CLASS:
                self.images.append((np.array(img), 1))  # Dog = class 1
        
        # Remove corrupted images (low standard deviation)
        if self.remove_corrupted:
            original_size = len(self.images)
            self.images = [
                (img, lbl) for img, lbl in self.images
                if np.std(img) >= self.CORRUPTION_THRESHOLD
            ]
            print(f"  Removed {original_size - len(self.images)} corrupted images")
        
        print(f"✓ Loaded {len(self.images)} {split} images (cats vs dogs)")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img, label = self.images[idx]
        img_pil = Image.fromarray(img)
        
        if self.transform:
            img = self.transform(img_pil)
        else:
            img = transforms.ToTensor()(img_pil)
        
        return img, label

print("✓ BinaryCIFAR10 dataset class defined")

In [ ]:
# Load and split datasets
print("Loading CIFAR-10 data...")
train_dataset = BinaryCIFAR10(split='train', transform=train_transform, remove_corrupted=True)

# Stratified train/val/test split (80/10/10)
total_size = len(train_dataset)
train_size = int(0.8 * total_size)  # 80%
val_size = int(0.1 * total_size)    # 10%
test_size = total_size - train_size - val_size  # 10%

# Get indices and shuffle
indices = list(range(total_size))
np.random.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:train_size+val_size]
test_indices = indices[train_size+val_size:]

# Create subsets
train_subset = torch.utils.data.Subset(train_dataset, train_indices)
val_subset = torch.utils.data.Subset(train_dataset, val_indices)
test_subset = torch.utils.data.Subset(train_dataset, test_indices)

# Create dataloaders
BATCH_SIZE = 64
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f"\nDataset splits:")
print(f"  Train: {len(train_subset)} images ({train_size/total_size*100:.1f}%)")
print(f"  Val:   {len(val_subset)} images ({val_size/total_size*100:.1f}%)")
print(f"  Test:  {len(test_subset)} images ({test_size/total_size*100:.1f}%)")
print(f"  Batch size: {BATCH_SIZE}")

## 2. Model Architecture

ResNet50 pretrained on ImageNet, adapted for binary classification (cats vs dogs).

**Architecture:**
- Backbone: ResNet50 (50-layer residual network)
- Input: 32×32×3 (CIFAR-10 images)
- Output: 2 classes (cat or dog)
- Custom head: 1000→512→256→2 with dropout and ReLU activations

In [ ]:
# ResNet50 model for binary classification
class ResNet50Binary(nn.Module):
    """ResNet50 pretrained on ImageNet, adapted for binary classification"""
    
    def __init__(self, num_classes=2, dropout_rate=0.5):
        super().__init__()
        
        # Load pretrained ResNet50
        resnet50 = models.resnet50(pretrained=True)
        
        # Remove the final classification layer (1000 classes)
        self.backbone = nn.Sequential(*list(resnet50.children())[:-1])
        
        # Custom head for binary classification
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        """Forward pass: backbone -> head -> logits"""
        features = self.backbone(x)
        logits = self.head(features)
        return logits

# Create model
model = ResNet50Binary(num_classes=2, dropout_rate=0.5).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"✓ ResNet50Binary model created")
print(f"  Total parameters: {total_params:,}")
print(f"  Trainable parameters: {trainable_params:,}")

In [ ]:
# Define training configuration
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001, weight_decay=0.0001)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3
)

# Early stopping configuration
class EarlyStoppingCallback:
    """Stop training if validation loss doesn't improve"""
    
    def __init__(self, patience=5, verbose=True):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_loss = None
        self.early_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            if self.verbose:
                print(f"✓ Validation loss improved to {val_loss:.4f}")
        else:
            self.counter += 1
            if self.verbose:
                print(f"  No improvement for {self.counter}/{self.patience} epochs")
            if self.counter >= self.patience:
                self.early_stop = True

early_stopping = EarlyStoppingCallback(patience=5, verbose=True)

print("✓ Training configuration set up")
print(f"  Loss function: CrossEntropyLoss + L2 (λ=0.0001)")
print(f"  Optimizer: Adam (lr=0.001, weight_decay=0.0001)")
print(f"  Learning rate scheduler: ReduceLROnPlateau")